In [1]:
import numpy as np
import pandas as pd
import scipy.sparse
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score, accuracy_score
from sklearn.model_selection import GridSearchCV


In [2]:
# ── Labels ───────────────────────────────────────────────────────────────────
y_train       = np.load('saved_features/y_train.npy')
y_val         = np.load('saved_features/y_val.npy')
y_train_no_sw = np.load('saved_features/y_train_no_sw.npy')
y_val_no_sw   = np.load('saved_features/y_val_no_sw.npy')
y_train_raw   = np.load('saved_features/y_train_raw.npy')
y_val_raw     = np.load('saved_features/y_val_raw.npy')

# ── Raw ──────────────────────────────────────────────────────────────────────
X_train_raw = np.load('saved_features/X_train_raw.npy', allow_pickle=True)
X_val_raw   = np.load('saved_features/X_val_raw.npy', allow_pickle=True)

# ── BoW ──────────────────────────────────────────────────────────────────────
X_train_bow       = scipy.sparse.load_npz('saved_features/X_train_bow.npz')
X_val_bow         = scipy.sparse.load_npz('saved_features/X_val_bow.npz')
X_train_bow_no_sw = scipy.sparse.load_npz('saved_features/X_train_bow_no_sw.npz')
X_val_bow_no_sw   = scipy.sparse.load_npz('saved_features/X_val_bow_no_sw.npz')

# ── TF-IDF ───────────────────────────────────────────────────────────────────
X_train_tfidf       = scipy.sparse.load_npz('saved_features/X_train_tfidf.npz')
X_val_tfidf         = scipy.sparse.load_npz('saved_features/X_val_tfidf.npz')
X_train_tfidf_no_sw = scipy.sparse.load_npz('saved_features/X_train_tfidf_no_sw.npz')
X_val_tfidf_no_sw   = scipy.sparse.load_npz('saved_features/X_val_tfidf_no_sw.npz')

# ── Word2Vec Skip-gram ────────────────────────────────────────────────────────
X_train_w2v       = np.load('saved_features/X_train_w2v.npy')
X_val_w2v         = np.load('saved_features/X_val_w2v.npy')
X_train_w2v_no_sw = np.load('saved_features/X_train_w2v_no_sw.npy')
X_val_w2v_no_sw   = np.load('saved_features/X_val_w2v_no_sw.npy')

# ── Word2Vec CBOW ─────────────────────────────────────────────────────────────
X_train_w2v_cbow       = np.load('saved_features/X_train_w2v_cbow.npy')
X_val_w2v_cbow         = np.load('saved_features/X_val_w2v_cbow.npy')
X_train_w2v_cbow_no_sw = np.load('saved_features/X_train_w2v_cbow_no_sw.npy')
X_val_w2v_cbow_no_sw   = np.load('saved_features/X_val_w2v_cbow_no_sw.npy')

# ── GloVe ─────────────────────────────────────────────────────────────────────
X_train_glove       = np.load('saved_features/X_train_glove.npy')
X_val_glove         = np.load('saved_features/X_val_glove.npy')
X_train_glove_no_sw = np.load('saved_features/X_train_glove_no_sw.npy')
X_val_glove_no_sw   = np.load('saved_features/X_val_glove_no_sw.npy')

# ── FinBERT ───────────────────────────────────────────────────────────────────
X_train_finbert = np.load('saved_features/X_train_finbert.npy')
X_val_finbert   = np.load('saved_features/X_val_finbert.npy')

# ── RoBERTa ───────────────────────────────────────────────────────────────────
X_train_roberta = np.load('saved_features/X_train_roberta.npy')
X_val_roberta   = np.load('saved_features/X_val_roberta.npy')

# ── BERTweet ──────────────────────────────────────────────────────────────────
X_train_bertweet = np.load('saved_features/X_train_bertweet.npy')
X_val_bertweet   = np.load('saved_features/X_val_bertweet.npy')

print(f"Train size: {len(y_train):,} tweets")
print(f"Validation size: {len(y_val):,} tweets")

Train size: 7,634 tweets
Validation size: 1,909 tweets


In [3]:
def train_evaluate_rf(X_train, X_val, y_train, y_val, label, n_estimators=200):
    
    rf = RandomForestClassifier(
        n_estimators=n_estimators,   # number of trees
        class_weight='balanced',     # to handle class imbalance
        random_state=42,
        n_jobs=-1                  
    )
    
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_val)
    
    print(f"\n{'='*55}")
    print(f"  Random Forest — {label}")
    print(f"{'='*55}")
    print(classification_report(y_val, y_pred,
                                target_names=['Bearish', 'Bullish', 'Neutral']))
    
    print('\nConfusion Matrix for Validation Data:')
    print(confusion_matrix(y_val, y_pred))
    
    return rf, y_pred

In [4]:
rf_bow, _ = train_evaluate_rf(X_train_bow, X_val_bow, y_train, y_val, 
                               label="BoW")

# TF-IDF
rf_tfidf, _ = train_evaluate_rf(X_train_tfidf, X_val_tfidf, y_train, y_val,
                                 label="TF-IDF")


  Random Forest — BoW
              precision    recall  f1-score   support

     Bearish       0.85      0.41      0.55       288
     Bullish       0.73      0.55      0.63       385
     Neutral       0.79      0.95      0.86      1236

    accuracy                           0.79      1909
   macro avg       0.79      0.64      0.68      1909
weighted avg       0.79      0.79      0.77      1909


Confusion Matrix for Validation Data:
[[ 117   25  146]
 [  10  212  163]
 [  11   52 1173]]

  Random Forest — TF-IDF
              precision    recall  f1-score   support

     Bearish       0.82      0.36      0.50       288
     Bullish       0.71      0.50      0.59       385
     Neutral       0.78      0.95      0.85      1236

    accuracy                           0.77      1909
   macro avg       0.77      0.61      0.65      1909
weighted avg       0.77      0.77      0.75      1909


Confusion Matrix for Validation Data:
[[ 105   27  156]
 [  11  194  180]
 [  12   52 1172]]


### test with word2vec, and glove

In [5]:
# Word2Vec Skip-gram
rf_w2v, _ = train_evaluate_rf(X_train_w2v, X_val_w2v, y_train, y_val,
                               label="Word2Vec Skip-gram")

# Word2Vec CBOW
rf_w2v_cbow, _ = train_evaluate_rf(X_train_w2v_cbow, X_val_w2v_cbow, y_train, y_val,
                                    label="Word2Vec CBOW")

# GloVe
rf_glove, _ = train_evaluate_rf(X_train_glove, X_val_glove, y_train, y_val,
                                 label="GloVe-100")


  Random Forest — Word2Vec Skip-gram
              precision    recall  f1-score   support

     Bearish       0.56      0.13      0.21       288
     Bullish       0.56      0.31      0.40       385
     Neutral       0.72      0.95      0.82      1236

    accuracy                           0.70      1909
   macro avg       0.61      0.46      0.48      1909
weighted avg       0.66      0.70      0.64      1909


Confusion Matrix for Validation Data:
[[  37   46  205]
 [  14  119  252]
 [  15   47 1174]]

  Random Forest — Word2Vec CBOW
              precision    recall  f1-score   support

     Bearish       0.62      0.07      0.13       288
     Bullish       0.59      0.18      0.28       385
     Neutral       0.68      0.97      0.80      1236

    accuracy                           0.68      1909
   macro avg       0.63      0.41      0.40      1909
weighted avg       0.66      0.68      0.59      1909


Confusion Matrix for Validation Data:
[[  21   16  251]
 [   9   70  306

In [6]:
rf_finbert, _ = train_evaluate_rf(X_train_finbert, X_val_finbert, y_train, y_val, label="FinBERT CLS")

rf_roberta, _ = train_evaluate_rf(X_train_roberta, X_val_roberta, y_train, y_val, label="RoBERTa CLS")

rf_bertweet, _ = train_evaluate_rf(X_train_bertweet, X_val_bertweet, y_train, y_val, label="BERTweet CLS")


  Random Forest — FinBERT CLS
              precision    recall  f1-score   support

     Bearish       0.78      0.61      0.69       288
     Bullish       0.79      0.56      0.66       385
     Neutral       0.82      0.94      0.87      1236

    accuracy                           0.81      1909
   macro avg       0.80      0.70      0.74      1909
weighted avg       0.81      0.81      0.80      1909


Confusion Matrix for Validation Data:
[[ 176   10  102]
 [  15  215  155]
 [  34   46 1156]]

  Random Forest — RoBERTa CLS
              precision    recall  f1-score   support

     Bearish       0.88      0.20      0.33       288
     Bullish       0.74      0.45      0.56       385
     Neutral       0.76      0.98      0.85      1236

    accuracy                           0.76      1909
   macro avg       0.79      0.55      0.58      1909
weighted avg       0.77      0.76      0.72      1909


Confusion Matrix for Validation Data:
[[  58   39  191]
 [   8  175  202]
 [   0 

In [7]:
def check_data_shapes(X_train, X_val, label):
    print(f"\n--- {label} ---")
    print(f"\tX_train shape: {X_train.shape}")
    print(f"\tX_val shape: {X_val.shape}")
    
    # Para ver valores únicos (funciona para numpy e sparse)
    if hasattr(X_train, 'toarray'):
        # É matriz esparsa - converte uma pequena amostra
        sample = X_train[:10].toarray()
        print(f"\tUnique values in X_train sample: {np.unique(sample)[:10]}...")
        print(f"\tAny all-zero rows? {(X_train.toarray() == 0).all(axis=1).sum()}")
    else:
        # É numpy array normal
        print(f"\tUnique values in X_train: {np.unique(X_train[:10])}")
        print(f"\tAny all-zero rows? {(X_train == 0).all(axis=1).sum()}")
    
    # Verifica variância dos dados (importante!)
    if hasattr(X_train, 'toarray'):
        data_sample = X_train[:100].toarray()
    else:
        data_sample = X_train[:100]
    
    print(f"\tStd deviation of data: {data_sample.std():.6f}")
    print(f"\tMean of data: {data_sample.mean():.6f}")

# Verifica cada modelo
check_data_shapes(X_train_finbert, X_val_finbert, "FinBERT")
check_data_shapes(X_train_bow, X_val_bow, "BoW")
check_data_shapes(X_train_w2v, X_val_w2v, "Word2Vec")
check_data_shapes(X_train_glove, X_val_glove, "GloVe")


--- FinBERT ---
	X_train shape: (7634, 768)
	X_val shape: (1909, 768)
	Unique values in X_train: [-2.918073  -2.8785074 -2.8449316 ...  2.1013067  2.2925577  3.3046448]
	Any all-zero rows? 0
	Std deviation of data: 0.608167
	Mean of data: -0.016211

--- BoW ---
	X_train shape: (7634, 5000)
	X_val shape: (1909, 5000)
	Unique values in X_train sample: [0 1 2]...
	Any all-zero rows? 19
	Std deviation of data: 0.047348
	Mean of data: 0.002050

--- Word2Vec ---
	X_train shape: (7634, 100)
	X_val shape: (1909, 100)
	Unique values in X_train: [-6.22123241e-01 -5.57573855e-01 -5.33129394e-01 -4.79754418e-01
 -4.60361660e-01 -4.50389445e-01 -4.47562605e-01 -4.12937254e-01
 -3.97563308e-01 -3.91025752e-01 -3.86852562e-01 -3.85098636e-01
 -3.81462634e-01 -3.80206078e-01 -3.77997607e-01 -3.57475072e-01
 -3.54732007e-01 -3.52262437e-01 -3.50946754e-01 -3.42617720e-01
 -3.41608465e-01 -3.34062487e-01 -3.31352711e-01 -3.27552795e-01
 -3.25447649e-01 -3.10788095e-01 -3.09007198e-01 -3.07624698e-01
 -

In [8]:
print(" GRID SEARCH - FINBERT CLS")

param_grid = {
    'n_estimators': [200, 300],   
    'max_depth': [20, None], 
    'min_samples_split': [2, 5],    
}

rf_base = RandomForestClassifier(
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)


grid_search = GridSearchCV(
    estimator=rf_base,
    param_grid=param_grid,
    cv=5,
    scoring='f1_macro', 
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_finbert, y_train)

print(f"\nBest params: {grid_search.best_params_}")
print(f"Best F1 macro (CV): {grid_search.best_score_:.4f}")

y_pred_best = grid_search.predict(X_val_finbert)

print("\n Results with best parameters:")
print(classification_report(y_val, y_pred_best, target_names=['Bearish', 'Bullish', 'Neutral']))

# Compare with original model
from sklearn.metrics import f1_score

# original model (default)
rf_default = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf_default.fit(X_train_finbert, y_train)
y_pred_default = rf_default.predict(X_val_finbert)

f1_default = f1_score(y_val, y_pred_default, average='macro')
f1_best = f1_score(y_val, y_pred_best, average='macro')

print(f"Comparison:")
print(f" Original Model (default): F1 macro = {f1_default:.4f}")
print(f" Optimized Model: F1 macro = {f1_best:.4f}")
print(f" Improvement:{f1_best - f1_default:+.4f}")

 GRID SEARCH - FINBERT CLS
Fitting 5 folds for each of 8 candidates, totalling 40 fits

Best params: {'max_depth': 20, 'min_samples_split': 5, 'n_estimators': 300}
Best F1 macro (CV): 0.7537

 Results with best parameters:
              precision    recall  f1-score   support

     Bearish       0.73      0.66      0.69       288
     Bullish       0.78      0.60      0.68       385
     Neutral       0.84      0.92      0.87      1236

    accuracy                           0.81      1909
   macro avg       0.78      0.73      0.75      1909
weighted avg       0.81      0.81      0.81      1909

Comparison:
 Original Model (default): F1 macro = 0.7381
 Optimized Model: F1 macro = 0.7496
 Improvement:+0.0115


In [9]:
# SIMPLE COMPARISON TABLE - Random Forest Models

data = {
    'Model': ['FinBERT CLS', 'RoBERTa CLS', 'BoW', 'Word2Vec Skip-gram'],
    'Accuracy': ['81%', '76%', '79%', '70%'],
    'F1 Macro': [0.74, 0.58, 0.68, 0.48],
    'F1 Bearish': [0.69, 0.33, 0.55, 0.21],
    'F1 Bullish': [0.66, 0.56, 0.63, 0.40],
    'F1 Neutral': [0.87, 0.85, 0.86, 0.82]
}

df = pd.DataFrame(data)
df = df.sort_values('F1 Macro', ascending=False)

print("\n" + "-"*80)
print("Performance Comparison of Random Forest Models")
print("-"*80)
print(df.to_string(index=False))
print("-"*80)


--------------------------------------------------------------------------------
Performance Comparison of Random Forest Models
--------------------------------------------------------------------------------
             Model Accuracy  F1 Macro  F1 Bearish  F1 Bullish  F1 Neutral
       FinBERT CLS      81%      0.74        0.69        0.66        0.87
               BoW      79%      0.68        0.55        0.63        0.86
       RoBERTa CLS      76%      0.58        0.33        0.56        0.85
Word2Vec Skip-gram      70%      0.48        0.21        0.40        0.82
--------------------------------------------------------------------------------


## Summary

**Best Model:** FinBERT CLS (81% accuracy, macro F1: 0.74)

**Key Findings:**
- FinBERT CLS outperforms all other models, especially on minority classes (Bearish F1: 0.69, Bullish F1: 0.66)
- BoW provides a strong baseline (79% accuracy, macro F1: 0.68)
- RoBERTa CLS fails to capture Bearish sentiment effectively (F1: 0.33)
- Word2Vec Skip-gram performs poorly across all classes (macro F1: 0.48)

**Optimization:** Grid Search improved FinBERT's macro F1 by +1.15% (0.738 → 0.750)

## Summary

**Best Model:** FinBERT CLS + Optimized RF

**Accuracy:** 81% | **Macro F1:** 0.75

**Bearish F1:** 0.69 | **Bullish F1:** 0.68 | **Neutral F1:** 0.87

**Improvement:** +5% Bearish recall, +4% Bullish recall after tuning

**Best params:** n_estimators=300, max_depth=20, min_samples_split=5